In [1]:
import joblib
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier

print("--- BƯỚC 1: ĐANG TẢI DỮ LIỆU & HUẤN LUYỆN MÔ HÌNH ---")

# 1. Tải dữ liệu 2 năm
usd_krw = yf.download("KRW=X", period="2y", interval="1d", auto_adjust=True)[
    "Close"
]
usd_vnd = yf.download("VND=X", period="2y", interval="1d", auto_adjust=True)[
    "Close"
]

if isinstance(usd_krw, pd.DataFrame):
  usd_krw = usd_krw.iloc[:, 0]
if isinstance(usd_vnd, pd.DataFrame):
  usd_vnd = usd_vnd.iloc[:, 0]

df = pd.DataFrame({"USD_KRW": usd_krw, "USD_VND": usd_vnd}).dropna()
raw_prices = df["USD_VND"] / df["USD_KRW"]

if raw_prices.index.tz is not None:
  raw_prices.index = raw_prices.index.tz_localize(None)
raw_prices.index = pd.to_datetime(raw_prices.index).normalize()


# 2. Hàm vi phân phân số (FFD) làm sạch dữ liệu
def get_weights_ffd(d, thres):
  w = [1.0]
  k = 1
  while True:
    w_ = -w[-1] / k * (d - k + 1)
    if abs(w_) < thres:
      break
    w.append(w_)
    k += 1
  return np.array(w[::-1])


def fractional_diff_ffd(series, d, thres=1e-4):
  w = get_weights_ffd(d, thres)
  width = len(w) - 1
  df_res = pd.Series(index=series.index)
  for iloc in range(width, len(series)):
    loc = series.index[iloc]
    sub_series = series.iloc[iloc - width : iloc + 1]
    df_res.loc[loc] = np.dot(w, sub_series.values)
  return df_res.dropna()


stationary_series = fractional_diff_ffd(raw_prices, d=0.05, thres=1e-4)

# 3. Tạo không gian đặc trưng và nhãn giả lập chuẩn
daily_vol = raw_prices.pct_change().ewm(span=30).std().dropna()
common_idx = (
    raw_prices.index.intersection(daily_vol.index)
    .intersection(stationary_series.index)
)

features_df = pd.DataFrame(index=common_idx)
features_df["Frac_Diff"] = stationary_series.loc[common_idx]
features_df["Rolling_Mean"] = features_df["Frac_Diff"].rolling(window=5).mean()
features_df["Rolling_Std"] = features_df["Frac_Diff"].rolling(window=5).std()
features_df["Return_1d"] = raw_prices.pct_change().loc[common_idx]

# Nhãn mục tiêu cơ sở (Dự đoán xu hướng 5 phiên tới)
features_df["Target_Return"] = raw_prices.pct_change(5).shift(-5).loc[common_idx]
features_df["Primary_Bin"] = np.where(features_df["Target_Return"] > 0, 1, 0)
features_df = features_df.dropna()

X = features_df[["Frac_Diff", "Rolling_Mean", "Rolling_Std", "Return_1d"]]
meta_y = features_df["Primary_Bin"]

# 4. Huấn luyện Random Forest và LƯU LẠI thành file cứng
trained_model = RandomForestClassifier(
    n_estimators=100, max_depth=3, random_state=42, class_weight="balanced"
)
trained_model.fit(X, meta_y)

# Lưu mô hình ra file để các ngày sau cứ thế dùng
joblib.dump(trained_model, "krw_vnd_meta_model.pkl")
print(
    "✅ ĐÃ HUẤN LUYỆN XONG VÀ LƯU MÔ HÌNH VÀO FILE 'krw_vnd_meta_model.pkl'!"
)

--- BƯỚC 1: ĐANG TẢI DỮ LIỆU & HUẤN LUYỆN MÔ HÌNH ---


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


✅ ĐÃ HUẤN LUYỆN XONG VÀ LƯU MÔ HÌNH VÀO FILE 'krw_vnd_meta_model.pkl'!


In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import sklearn.model_selection as ms
import yfinance as yf

print("==================================================")
print(" BẮT ĐẦU CHẠY HỆ THỐNG ĐỊNH LƯỢNG KRW/VND (ALL-IN-ONE)")
print("==================================================")

# ==========================================
# 1. TẢI DỮ LIỆU & CHUẨN HÓA MÚI GIỜ
# ==========================================
print("\n--- [1/5] Đang tải dữ liệu từ Yahoo Finance ---")
usd_krw = yf.download("KRW=X", period="2y", interval="1d", auto_adjust=True)[
    "Close"
]
usd_vnd = yf.download("VND=X", period="2y", interval="1d", auto_adjust=True)[
    "Close"
]

if isinstance(usd_krw, pd.DataFrame):
  usd_krw = usd_krw.iloc[:, 0]
if isinstance(usd_vnd, pd.DataFrame):
  usd_vnd = usd_vnd.iloc[:, 0]

df = pd.DataFrame({"USD_KRW": usd_krw, "USD_VND": usd_vnd}).dropna()
raw_prices = df["USD_VND"] / df["USD_KRW"]

if raw_prices.index.tz is not None:
  raw_prices.index = raw_prices.index.tz_localize(None)
raw_prices.index = pd.to_datetime(raw_prices.index).normalize()


# ==========================================
# 2. BƯỚC 1: VI PHÂN PHÂN SỐ CỬA SỔ CỐ ĐỊNH (FFD, d = 0.05)
# ==========================================
print("--- [2/5] Đang xử lý FFD (Làm sạch dữ liệu giữ lại ký ức) ---")


def get_weights_ffd(d, thres):
  w = [1.0]
  k = 1
  while True:
    w_ = -w[-1] / k * (d - k + 1)
    if abs(w_) < thres:
      break
    w.append(w_)
    k += 1
  return np.array(w[::-1])


def fractional_diff_ffd(series, d, thres=1e-4):
  w = get_weights_ffd(d, thres)
  width = len(w) - 1
  df_res = pd.Series(index=series.index)
  for iloc in range(width, len(series)):
    loc = series.index[iloc]
    sub_series = series.iloc[iloc - width : iloc + 1]
    df_res.loc[loc] = np.dot(w, sub_series.values)
  return df_res.dropna()


stationary_series = fractional_diff_ffd(raw_prices, d=0.05, thres=1e-4)


# ==========================================
# 3. BƯỚC 2: TRIPLE BARRIER METHOD & LABELS
# ==========================================
print("--- [3/5] Đang gán nhãn Rào cản ba lớp (Triple Barrier) ---")


def get_daily_vol(close, span=30):
  return close.pct_change().ewm(span=span).std().dropna()


daily_vol = get_daily_vol(raw_prices)

common_idx = (
    raw_prices.index.intersection(daily_vol.index)
    .intersection(stationary_series.index)
)
raw_prices_clean = raw_prices.loc[common_idx]
daily_vol_clean = daily_vol.loc[common_idx]
stat_clean = stationary_series.loc[common_idx]


def apply_triple_barrier(prices_series, events, pt_sl, min_ret=0.001):
  out = pd.DataFrame(index=events.index)
  out["ret"] = np.nan
  out["bin"] = 0
  for loc, t1 in events["t1"].items():
    path = prices_series.loc[loc:t1]
    if len(path) <= 1:
      continue
    base_val = prices_series.loc[loc]
    if base_val == 0 or pd.isna(base_val):
      continue
    ret = (path / base_val) - 1
    target = max(events.loc[loc, "trgt"], min_ret)
    pt = target * pt_sl[0]
    sl = -target * pt_sl[1]
    hit_pt = ret[ret >= pt].index.min()
    hit_sl = ret[ret <= sl].index.min()
    if pd.notna(hit_pt) and (pd.isna(hit_sl) or hit_pt < hit_sl):
      out.loc[loc, "bin"] = 1
      out.loc[loc, "ret"] = ret.loc[hit_pt]
    elif pd.notna(hit_sl) and (pd.isna(hit_pt) or hit_sl < hit_pt):
      out.loc[loc, "bin"] = -1
      out.loc[loc, "ret"] = ret.loc[hit_sl]
    else:
      out.loc[loc, "bin"] = 0
      out.loc[loc, "ret"] = ret.iloc[-1]
  return out.dropna(subset=["ret"])


holding_period = 5
events_index = raw_prices_clean.index[30:-holding_period]
events = pd.DataFrame(index=events_index)
events["trgt"] = daily_vol_clean.loc[events.index]
events["t1"] = raw_prices_clean.index[
    [
        min(
            raw_prices_clean.index.get_loc(idx) + holding_period,
            len(raw_prices_clean) - 1,
        )
        for idx in events.index
    ]
]

labels = apply_triple_barrier(
    raw_prices_clean, events, pt_sl=[1.0, 1.0], min_ret=0.0005
)

# ==========================================
# 4. BƯỚC 3: TẬP ĐẶC TRƯNG & META-MODEL
# ==========================================
print("--- [4/5] Đang huấn luyện mô hình & Kiểm định chéo ---")
features_df = pd.DataFrame(index=labels.index)
features_df["Frac_Diff"] = stat_clean.loc[labels.index]
features_df["Rolling_Mean"] = features_df["Frac_Diff"].rolling(window=5).mean()
features_df["Rolling_Std"] = features_df["Frac_Diff"].rolling(window=5).std()
features_df["Return_1d"] = raw_prices_clean.pct_change().loc[labels.index]
features_df["Primary_Bin"] = labels["bin"]

features_df = features_df.dropna()

X = features_df[["Frac_Diff", "Rolling_Mean", "Rolling_Std", "Return_1d"]]
y_primary = features_df["Primary_Bin"]
meta_y = (y_primary == 1).astype(int)


# Purged K-Fold chuẩn López de Prado
class PurgedKFold(ms.KFold):

  def __init__(self, n_splits=5, t1=None, embargo_pct=0.01):
    super(PurgedKFold, self).__init__(
        n_splits=n_splits, shuffle=False, random_state=None
    )
    self.t1 = t1
    self.embargo_pct = embargo_pct

  def split(self, X, y=None, groups=None):
    if self.t1 is None:
      raise ValueError("Cần chuỗi thời gian t1")
    for train_index, test_index in super().split(X, y):
      test_start = X.index[test_index[0]]
      test_end = X.index[test_index[-1]]
      train_times = self.t1.loc[X.index[train_index]]
      banned_indices = train_times[
          (train_times >= test_start) & (train_times <= test_end)
      ].index
      train_index = np.array(
          [i for i in train_index if X.index[i] not in banned_indices]
      )
      embargo_idx = int(X.shape[0] * self.embargo_pct)
      if embargo_idx > 0:
        test_end_loc = X.index.get_loc(test_end)
        embargo_end_loc = min(test_end_loc + embargo_idx, X.shape[0])
        embargo_dates = X.index[test_end_loc:embargo_end_loc]
        train_index = np.array(
            [i for i in train_index if X.index[i] not in embargo_dates]
        )
      yield train_index, test_index


t1_series = events["t1"].loc[X.index]
cv = PurgedKFold(n_splits=5, t1=t1_series, embargo_pct=0.01)
final_model = RandomForestClassifier(
    n_estimators=100, max_depth=3, random_state=42, class_weight="balanced"
)

accuracies = []
for train_idx, test_idx in cv.split(X):
  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = meta_y.iloc[train_idx], meta_y.iloc[test_idx]
  final_model.fit(X_train, y_train)
  preds = final_model.predict(X_test)
  accuracies.append(accuracy_score(y_test, preds))

print(
    f" -> Độ chính xác trung bình kiểm định chéo (CV Accuracy):"
    f" {np.mean(accuracies)*100:.2f}%"
)

# Huấn luyện mô hình cuối cùng trên toàn bộ dữ liệu sạch
final_model.fit(X, meta_y)


# ==========================================
# 5. DỰ ĐOÁN TÍN HIỆU NGAY HÔM NAY (LIVE INFERENCE)
# ==========================================
print("--- [5/5] Đang quét dữ liệu phiên mới nhất để lấy tín hiệu ---")

latest_features = pd.DataFrame(index=raw_prices.index)
latest_features["Frac_Diff"] = stationary_series
latest_features["Rolling_Mean"] = latest_features["Frac_Diff"].rolling(
    window=5
).mean()
latest_features["Rolling_Std"] = latest_features["Frac_Diff"].rolling(
    window=5
).std()
latest_features["Return_1d"] = raw_prices.pct_change()

clean_features = latest_features.dropna()
current_data = clean_features.iloc[[-1]]
latest_date = current_data.index[0].strftime("%Y-%m-%d")
current_price = raw_prices.loc[current_data.index[0]]

pred_prob = final_model.predict_proba(
    current_data[["Frac_Diff", "Rolling_Mean", "Rolling_Std", "Return_1d"]]
)[:, 1][0]

print("\n" + "=" * 50)
print(f" BẢN TIN KẾT QUẢ GIAO DỊCH NGÀY: {latest_date}")
print("=" * 50)
print(f" Tỷ giá KRW/VND hiện tại: {current_price:.4f}")
print(f" Xác suất thành công (Meta-Probability): {pred_prob * 100:.2f}%")

if pred_prob > 0.5:
  print(" 🟢 TÍN HIỆU: ĐÈN XANH (Thực thi mở vị thế theo Rào cản ba lớp)")
else:
  print(" 🔴 TÍN HIỆU: ĐÈN ĐỎ (Thị trường rủi ro / Đứng ngoài, bảo toàn vốn)")
print("=" * 50)

 BẮT ĐẦU CHẠY HỆ THỐNG ĐỊNH LƯỢNG KRW/VND (ALL-IN-ONE)

--- [1/5] Đang tải dữ liệu từ Yahoo Finance ---


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


--- [2/5] Đang xử lý FFD (Làm sạch dữ liệu giữ lại ký ức) ---
--- [3/5] Đang gán nhãn Rào cản ba lớp (Triple Barrier) ---
--- [4/5] Đang huấn luyện mô hình & Kiểm định chéo ---
 -> Độ chính xác trung bình kiểm định chéo (CV Accuracy): 61.85%
--- [5/5] Đang quét dữ liệu phiên mới nhất để lấy tín hiệu ---

 BẢN TIN KẾT QUẢ GIAO DỊCH NGÀY: 2026-11-03
 Tỷ giá KRW/VND hiện tại: 19.1693
 Xác suất thành công (Meta-Probability): 45.07%
 🔴 TÍN HIỆU: ĐÈN ĐỎ (Thị trường rủi ro / Đứng ngoài, bảo toàn vốn)
